# Week 10, Class 13: Explainable AI - GradCAM for U-Net

**AI in Medicine and Healthcare**  
Prof. Dr. Marcel P. Jackowski  
Insper

---

## Learning Objectives

By the end of this lab, you will be able to:

1. Understand why explainability matters in medical AI
2. Implement GradCAM for segmentation models
3. Visualize which input regions influenced U-Net predictions
4. Interpret GradCAM heatmaps correctly
5. Identify when models are "right for wrong reasons"
6. Debug model errors using explainability

**Today's Goal:** Understand what your U-Net "sees" when making segmentations!

---

## Student Information

**Student 1:** _______________  

**Student 2:** _______________  


---

# PART 1: Setup

## Load Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import cv2
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print('✓ Libraries imported successfully!')

## Quick Recap: What We Did Last Class

Last class, you trained a U-Net for medical image segmentation:

1. **Dataset:** Synthetic images with circles and squares
2. **Models:** Compress-Expand, Naive CNN, U-Net
3. **Results:** U-Net achieved ~0.95 Dice score
4. **Question:** But WHY did it work so well?

Today we'll answer that question using GradCAM! 🔍

---

# PART 2: Load Trained U-Net

First, we need to recreate the U-Net architecture and load the trained weights.

## Define U-Net Architecture

In [ ]:
class UNet(nn.Module):
    """
    U-Net architecture for image segmentation
    """
    
    def __init__(self):
        super(UNet, self).__init__()
        
        # Encoder (downsampling path)
        self.enc1_1 = nn.Conv2d(1, 32, 3, padding=1)
        self.enc1_2 = nn.Conv2d(32, 32, 3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        self.enc2_1 = nn.Conv2d(32, 64, 3, padding=1)
        self.enc2_2 = nn.Conv2d(64, 64, 3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        self.enc3_1 = nn.Conv2d(64, 128, 3, padding=1)
        self.enc3_2 = nn.Conv2d(128, 128, 3, padding=1)
        self.pool3 = nn.MaxPool2d(2, 2)
        
        # Bottleneck
        self.bottleneck_1 = nn.Conv2d(128, 256, 3, padding=1)
        self.bottleneck_2 = nn.Conv2d(256, 256, 3, padding=1)
        
        # Decoder (upsampling path)
        self.up1 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec1_1 = nn.Conv2d(256 + 128, 128, 3, padding=1)
        self.dec1_2 = nn.Conv2d(128, 128, 3, padding=1)
        
        self.up2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec2_1 = nn.Conv2d(128 + 64, 64, 3, padding=1)
        self.dec2_2 = nn.Conv2d(64, 64, 3, padding=1)
        
        self.up3 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec3_1 = nn.Conv2d(64 + 32, 32, 3, padding=1)
        self.dec3_2 = nn.Conv2d(32, 32, 3, padding=1)
        
        # Output layer
        self.out = nn.Conv2d(32, 1, 1)
    
    def forward(self, x):
        # Encoder
        enc1 = F.relu(self.enc1_1(x))
        enc1 = F.relu(self.enc1_2(enc1))
        
        enc2 = self.pool1(enc1)
        enc2 = F.relu(self.enc2_1(enc2))
        enc2 = F.relu(self.enc2_2(enc2))
        
        enc3 = self.pool2(enc2)
        enc3 = F.relu(self.enc3_1(enc3))
        enc3 = F.relu(self.enc3_2(enc3))
        
        # Bottleneck
        x = self.pool3(enc3)
        x = F.relu(self.bottleneck_1(x))
        x = F.relu(self.bottleneck_2(x))
        
        # Decoder with skip connections
        x = self.up1(x)
        x = torch.cat([x, enc3], dim=1)
        x = F.relu(self.dec1_1(x))
        x = F.relu(self.dec1_2(x))
        
        x = self.up2(x)
        x = torch.cat([x, enc2], dim=1)
        x = F.relu(self.dec2_1(x))
        x = F.relu(self.dec2_2(x))
        
        x = self.up3(x)
        x = torch.cat([x, enc1], dim=1)
        x = F.relu(self.dec3_1(x))
        x = F.relu(self.dec3_2(x))
        
        # Output
        x = self.out(x)
        x = torch.sigmoid(x)
        
        return x

# Create model
model_unet = UNet().to(device)
print('✓ U-Net architecture loaded!')
print(f'  Total parameters: {sum(p.numel() for p in model_unet.parameters()):,}')

## Option 1: Load Your Trained Model

If you saved your model from last class, load it here:

```python
# Uncomment if you have a saved model
# model_unet.load_state_dict(torch.load('unet_trained.pth', map_location=device))
# print('✓ Loaded your trained U-Net!')
```

## Option 2: Train a Quick Model

If you don't have a saved model, let's first create a simple dataset and train a quick one:

In [ ]:
# For demonstration, we'll create a simple dataset and train quickly
# (In practice, you'd load your trained model from last class)

from scipy.ndimage import gaussian_filter

def create_quick_dataset(n_samples=100, img_size=128):
    """Create a small dataset for demonstration"""
    images = []
    masks = []
    
    for _ in range(n_samples):
        img = np.zeros((img_size, img_size))
        mask = np.zeros((img_size, img_size))
        
        # Random number of shapes
        n_shapes = np.random.randint(3, 6)
        
        for _ in range(n_shapes):
            size = np.random.randint(15, 40)
            x = np.random.randint(size, img_size-size)
            y = np.random.randint(size, img_size-size)
            
            if np.random.random() > 0.5:
                # Circle
                yy, xx = np.ogrid[:img_size, :img_size]
                circle = (xx - x)**2 + (yy - y)**2 <= (size//2)**2
                intensity = np.random.uniform(0.5, 1.0)
                img[circle] = intensity
                mask[circle] = 1
            else:
                # Square
                intensity = np.random.uniform(0.5, 1.0)
                img[y-size//2:y+size//2, x-size//2:x+size//2] = intensity
                mask[y-size//2:y+size//2, x-size//2:x+size//2] = 1
        
        # Add noise and blur
        img = gaussian_filter(img, sigma=1.5)
        noise = np.random.normal(0, 0.2, img.shape)
        img = np.clip(img + noise, 0, 1)
        
        if img.max() > 0:
            img = (img - img.min()) / (img.max() - img.min())
        
        images.append(img)
        masks.append(mask)
    
    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)

print('Generating quick dataset for demonstration...')
X_demo, y_demo = create_quick_dataset(n_samples=100)
print(f'✓ Created {len(X_demo)} samples')

# Visualize a few
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i in range(3):
    axes[0, i].imshow(X_demo[i], cmap='gray')
    axes[0, i].set_title(f'Input {i+1}')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(y_demo[i], cmap='gray')
    axes[1, i].set_title(f'Mask {i+1}')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

print('\n💡 Tip: If you have your trained model from last class, load it instead!')
print('   This quick dataset is just for demonstration.')

## Quick Training (Optional)

If needed, train the model quickly:

In [ ]:
# Quick training function
def quick_train(model, X, y, epochs=10):
    """Quick training for demonstration"""
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCELoss()
    
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for i in range(len(X)):
            img = torch.from_numpy(X[i:i+1]).unsqueeze(1).to(device)
            mask = torch.from_numpy(y[i:i+1]).unsqueeze(1).to(device)
            
            optimizer.zero_grad()
            output = model(img)
            loss = criterion(output, mask)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        if (epoch + 1) % 2 == 0:
            print(f'Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(X):.4f}')
    
    print('✓ Quick training complete!')
    return model

# Uncomment to train:
# model_unet = quick_train(model_unet, X_demo, y_demo, epochs=10)

print('⚠️  Skipping training for now - we assume you have a trained model.')
print('   Uncomment the line above if you need to train.')

** At this point, you should have a trained U-Net model ready!**

Either:
- ✅ Loaded from last class
- ✅ Or quickly trained just now

---

# PART 3: Implement GradCAM

**Goal:** Visualize which input regions influenced the segmentation

**How it works:**

1. **Forward pass:** Run image through U-Net → get segmentation
2. **Target selection:** Choose a specific output pixel (or region) to explain
3. **Backward pass:** Compute gradients from that pixel back to a target layer
4. **Weight calculation:** Average gradients across spatial dimensions → weights
5. **Weighted combination:** Multiply feature maps by weights and sum
6. **ReLU:** Keep only positive influences
7. **Upsample:** Resize to input size → GradCAM heatmap!

---

## Step 1: GradCAM Implementation

In [ ]:
class GradCAM:
    """
    GradCAM implementation for CNNs
    
    Generates heatmaps showing which input regions were important
    for a specific output.
    """
    
    def __init__(self, model, target_layer):
        """
        Parameters:
        -----------
        model : nn.Module
            The model to explain
        target_layer : nn.Module
            The layer to compute GradCAM from (usually last conv layer)
        """
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Register hooks to capture activations and gradients
        self.target_layer.register_forward_hook(self._save_activation)
        self.target_layer.register_backward_hook(self._save_gradient)
    
    def _save_activation(self, module, input, output):
        """Hook to capture forward activations"""
        self.activations = output.detach()
    
    def _save_gradient(self, module, grad_input, grad_output):
        """Hook to capture backward gradients"""
        self.gradients = grad_output[0].detach()
    
    def generate_cam(self, input_image, target_mask=None):
        """
        Generate GradCAM heatmap
        
        Parameters:
        -----------
        input_image : torch.Tensor
            Input image [1, C, H, W]
        target_mask : torch.Tensor, optional
            If provided, explains this specific segmentation mask
            If None, explains the entire output
        
        Returns:
        --------
        cam : numpy.ndarray
            GradCAM heatmap [H, W]
        """
        # Forward pass
        self.model.eval()
        output = self.model(input_image)
        
        # If no target mask provided, use the model's output
        if target_mask is None:
            target_mask = output
        
        # Backward pass
        self.model.zero_grad()
        
        # Compute gradient of target w.r.t. target layer
        # We want to explain: "Why did the model predict this mask?"
        # So we backprop from the mask prediction
        loss = (output * target_mask).sum()
        loss.backward()
        
        # Get gradients and activations
        gradients = self.gradients  # [1, C, H, W]
        activations = self.activations  # [1, C, H, W]
        
        # Global average pooling of gradients → weights
        weights = gradients.mean(dim=(2, 3), keepdim=True)  # [1, C, 1, 1]
        
        # Weighted combination of activation maps
        cam = (weights * activations).sum(dim=1, keepdim=True)  # [1, 1, H, W]
        
        # Apply ReLU (keep only positive influences)
        cam = F.relu(cam)
        
        # Normalize to [0, 1]
        cam = cam.squeeze().cpu().numpy()
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        
        # Upsample to input size
        cam = cv2.resize(cam, (input_image.shape[3], input_image.shape[2]))
        
        return cam

print('✓ GradCAM class defined!')

## Step 2: Choose Target Layer

For U-Net, we typically use the **bottleneck** or **last encoder layer** as the target.

Why? These layers have:
- Rich semantic information (what objects are present)
- Reasonable spatial resolution (where objects are)
- Best balance between "what" and "where"

Let's create GradCAM for the bottleneck:

In [ ]:
# Create GradCAM instance targeting the bottleneck
# bottleneck_2 is the last layer in the bottleneck
gradcam = GradCAM(model_unet, target_layer=model_unet.bottleneck_2)

print('✓ GradCAM initialized!')
print(f'  Target layer: bottleneck_2')
print(f'  This layer has shape: [batch, 256 channels, 16×16]')
print(f'  It contains high-level semantic features')

## Step 3: Visualization Helper Functions

Let's create functions to visualize GradCAM results beautifully:

In [ ]:
def overlay_heatmap(image, heatmap, alpha=0.4, colormap=cv2.COLORMAP_JET):
    """
    Overlay heatmap on original image
    
    Parameters:
    -----------
    image : numpy.ndarray
        Original image [H, W] (grayscale)
    heatmap : numpy.ndarray
        GradCAM heatmap [H, W]
    alpha : float
        Transparency of heatmap overlay
    colormap : int
        OpenCV colormap to use
    
    Returns:
    --------
    overlay : numpy.ndarray
        Image with heatmap overlay [H, W, 3]
    """
    # Normalize image to [0, 255]
    image = ((image - image.min()) / (image.max() - image.min()) * 255).astype(np.uint8)
    
    # Normalize heatmap to [0, 255]
    heatmap = (heatmap * 255).astype(np.uint8)
    
    # Apply colormap to heatmap
    heatmap_colored = cv2.applyColorMap(heatmap, colormap)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    
    # Convert grayscale to RGB
    image_rgb = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    
    # Overlay
    overlay = cv2.addWeighted(image_rgb, 1 - alpha, heatmap_colored, alpha, 0)
    
    return overlay

def visualize_gradcam_result(image, mask, prediction, gradcam_heatmap, title=''):
    """
    Comprehensive visualization of GradCAM results
    
    Shows: Input | Ground Truth | Prediction | GradCAM | Overlay
    """
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    
    # Input image
    axes[0].imshow(image, cmap='gray')
    axes[0].set_title('Input Image', fontsize=12)
    axes[0].axis('off')
    
    # Ground truth mask
    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title('Ground Truth', fontsize=12)
    axes[1].axis('off')
    
    # Prediction
    axes[2].imshow(prediction, cmap='gray', vmin=0, vmax=1)
    axes[2].set_title('U-Net Prediction', fontsize=12)
    axes[2].axis('off')
    
    # GradCAM heatmap
    im = axes[3].imshow(gradcam_heatmap, cmap='jet', vmin=0, vmax=1)
    axes[3].set_title('GradCAM Heatmap', fontsize=12)
    axes[3].axis('off')
    plt.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)
    
    # Overlay
    overlay = overlay_heatmap(image, gradcam_heatmap, alpha=0.5)
    axes[4].imshow(overlay)
    axes[4].set_title('Input + GradCAM', fontsize=12)
    axes[4].axis('off')
    
    if title:
        plt.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    plt.show()

print('✓ Visualization functions defined!')

---

# PART 4: Apply GradCAM to Test Images

## Generate Test Images

In [ ]:
# Generate a few test images
print('Generating test images...')
X_test, y_test = create_quick_dataset(n_samples=10, img_size=128)

# Get predictions
model_unet.eval()
predictions = []

with torch.no_grad():
    for i in range(len(X_test)):
        img = torch.from_numpy(X_test[i:i+1]).unsqueeze(1).to(device)
        pred = model_unet(img).cpu().numpy()[0, 0]
        predictions.append(pred)

predictions = np.array(predictions)

print(f'✓ Generated {len(X_test)} test images')
print(f'✓ Got predictions')
print(f'\nNow let\'s apply GradCAM to understand these predictions! 🔍')

## Apply GradCAM to Example 1

Let's analyze our first test image:

In [ ]:
# Select first test image
idx = 0
test_image = X_test[idx]
test_mask = y_test[idx]
test_pred = predictions[idx]

print(f'Analyzing test image {idx}...')
print(f'  Input shape: {test_image.shape}')
print(f'  Prediction range: [{test_pred.min():.3f}, {test_pred.max():.3f}]')

# Prepare input for GradCAM
input_tensor = torch.from_numpy(test_image).unsqueeze(0).unsqueeze(0).to(device)
input_tensor.requires_grad = True

# Generate GradCAM heatmap
print('\nGenerating GradCAM heatmap...')
cam = gradcam.generate_cam(input_tensor)

print(f'✓ GradCAM heatmap generated!')
print(f'  Heatmap shape: {cam.shape}')
print(f'  Heatmap range: [{cam.min():.3f}, {cam.max():.3f}]')

# Visualize
visualize_gradcam_result(
    test_image, 
    test_mask, 
    test_pred, 
    cam,
    title=f'GradCAM Analysis - Test Image {idx}'
)

### Questions to Consider:

Look at the visualization above and think about:

1. **Does the GradCAM heatmap align with the object boundaries?**
   - Red/yellow regions = high importance
   - Blue regions = low importance

2. **Is the model focusing on the right features?**
   - Should focus on the shapes (circles/squares)
   - Should NOT focus on background noise

3. **Do the important regions match the segmentation output?**
   - Compare GradCAM heatmap with prediction

**Discussion:** What do you observe? Does the model look at the right places?

---

## Apply GradCAM to Multiple Examples

Let's analyze several images to see patterns:

In [ ]:
# Analyze multiple test images
n_examples = 4
indices = np.random.choice(len(X_test), n_examples, replace=False)

for idx in indices:
    print(f'\n{"="*60}')
    print(f'Analyzing Test Image {idx}')
    print("="*60)
    
    # Get image, mask, prediction
    image = X_test[idx]
    mask = y_test[idx]
    pred = predictions[idx]
    
    # Calculate Dice score
    pred_binary = (pred > 0.5).astype(np.float32)
    intersection = (pred_binary * mask).sum()
    dice = (2. * intersection) / (pred_binary.sum() + mask.sum() + 1e-8)
    
    print(f'Dice Score: {dice:.4f}')
    
    # Generate GradCAM
    input_tensor = torch.from_numpy(image).unsqueeze(0).unsqueeze(0).to(device)
    input_tensor.requires_grad = True
    cam = gradcam.generate_cam(input_tensor)
    
    # Visualize
    visualize_gradcam_result(
        image, mask, pred, cam,
        title=f'Test Image {idx} | Dice: {dice:.4f}'
    )
    
    # Analysis
    # Check if GradCAM focuses on foreground
    fg_mask = mask > 0.5
    bg_mask = mask <= 0.5
    
    if fg_mask.sum() > 0 and bg_mask.sum() > 0:
        cam_fg_mean = cam[fg_mask].mean()
        cam_bg_mean = cam[bg_mask].mean()
        
        print(f'GradCAM Analysis:')
        print(f'  Average attention on foreground: {cam_fg_mean:.3f}')
        print(f'  Average attention on background: {cam_bg_mean:.3f}')
        print(f'  Ratio (fg/bg): {cam_fg_mean/cam_bg_mean:.2f}x')
        
        if cam_fg_mean > cam_bg_mean * 1.5:
            print('  ✓ Model focuses more on foreground (Good!)')
        else:
            print('  ⚠️  Model attention not strongly on foreground')

### What to Look For:

When examining these GradCAM visualizations, check for:

**Good Signs (✓):**
- Heatmap is bright (red/yellow) on object boundaries
- Heatmap is dark (blue) on background
- Attention aligns with ground truth mask
- Foreground/background ratio > 1.5x

**Red Flags (⚠️):**
- Heatmap bright on background/noise
- Heatmap dark on actual objects
- Random scattered hotspots
- Equal attention on foreground and background

---

# PART 5: Comparative Analysis

Let's compare GradCAM for **good predictions** vs **bad predictions**

## Find Best and Worst Predictions

In [ ]:
# Calculate Dice scores for all test images
dice_scores = []

for i in range(len(X_test)):
    pred_binary = (predictions[i] > 0.5).astype(np.float32)
    mask = y_test[i]
    intersection = (pred_binary * mask).sum()
    dice = (2. * intersection) / (pred_binary.sum() + mask.sum() + 1e-8)
    dice_scores.append(dice)

dice_scores = np.array(dice_scores)

# Find best and worst
best_idx = np.argmax(dice_scores)
worst_idx = np.argmin(dice_scores)

print('Test Set Performance:')
print(f'  Mean Dice: {dice_scores.mean():.4f}')
print(f'  Std Dice:  {dice_scores.std():.4f}')
print(f'  Min Dice:  {dice_scores.min():.4f} (image {worst_idx})')
print(f'  Max Dice:  {dice_scores.max():.4f} (image {best_idx})')
print(f'\nLet\'s compare these two extremes!')

## Compare Best vs. Worst

In [ ]:
# Compare best and worst predictions
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for row, (idx, label) in enumerate([(best_idx, 'BEST'), (worst_idx, 'WORST')]):
    image = X_test[idx]
    mask = y_test[idx]
    pred = predictions[idx]
    dice = dice_scores[idx]
    
    # Generate GradCAM
    input_tensor = torch.from_numpy(image).unsqueeze(0).unsqueeze(0).to(device)
    input_tensor.requires_grad = True
    cam = gradcam.generate_cam(input_tensor)
    
    # Plot
    axes[row, 0].imshow(image, cmap='gray')
    axes[row, 0].set_title(f'{label}\nInput', fontsize=11, fontweight='bold')
    axes[row, 0].axis('off')
    
    axes[row, 1].imshow(mask, cmap='gray')
    axes[row, 1].set_title('Ground Truth', fontsize=11)
    axes[row, 1].axis('off')
    
    axes[row, 2].imshow(pred, cmap='gray', vmin=0, vmax=1)
    axes[row, 2].set_title(f'Prediction\nDice: {dice:.3f}', fontsize=11)
    axes[row, 2].axis('off')
    
    axes[row, 3].imshow(cam, cmap='jet', vmin=0, vmax=1)
    axes[row, 3].set_title('GradCAM', fontsize=11)
    axes[row, 3].axis('off')
    
    overlay = overlay_heatmap(image, cam, alpha=0.5)
    axes[row, 4].imshow(overlay)
    axes[row, 4].set_title('Overlay', fontsize=11)
    axes[row, 4].axis('off')

plt.suptitle('Comparison: Best vs Worst Predictions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 🔍 Analysis Questions:

Compare the two rows above:

1. **Best Prediction (Top Row):**
   - Where does GradCAM show high attention?
   - Does it align with the objects?
   - Is the attention pattern clear and focused?

2. **Worst Prediction (Bottom Row):**
   - Where does GradCAM show high attention?
   - Is the attention scattered or unfocused?
   - Does it focus on irrelevant regions?

3. **Key Insight:**
   - Can GradCAM help us understand WHY the worst prediction failed?
   - What would you do to improve the model based on this analysis?

**💡 This is the power of explainability - it helps debug model failures!**

---

# PART 6: Exploring Different Target Layers

GradCAM can be applied to different layers. Let's see how the choice of layer affects the heatmap!

## Compare Multiple Layers

In [ ]:
# Create GradCAM for different layers
layers_to_try = {
    'Encoder Layer 3': model_unet.enc3_2,
    'Bottleneck': model_unet.bottleneck_2,
    'Decoder Layer 1': model_unet.dec1_2,
}

# Select a test image
idx = 0
image = X_test[idx]
mask = y_test[idx]
pred = predictions[idx]

# Generate GradCAM for each layer
fig, axes = plt.subplots(1, len(layers_to_try) + 1, figsize=(20, 4))

# Show input
axes[0].imshow(image, cmap='gray')
axes[0].set_title('Input Image', fontsize=12, fontweight='bold')
axes[0].axis('off')

# Generate and show GradCAM for each layer
for i, (layer_name, layer) in enumerate(layers_to_try.items()):
    # Create GradCAM instance
    gradcam_layer = GradCAM(model_unet, target_layer=layer)
    
    # Generate heatmap
    input_tensor = torch.from_numpy(image).unsqueeze(0).unsqueeze(0).to(device)
    input_tensor.requires_grad = True
    cam = gradcam_layer.generate_cam(input_tensor)
    
    # Plot
    overlay = overlay_heatmap(image, cam, alpha=0.6)
    axes[i+1].imshow(overlay)
    axes[i+1].set_title(layer_name, fontsize=12)
    axes[i+1].axis('off')

plt.suptitle('GradCAM from Different Layers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📊 Observations:')
print('  - Earlier layers (Encoder): Focus on low-level features (edges, textures)')
print('  - Bottleneck: Balance of semantic + spatial info (usually best!)')
print('  - Decoder layers: More diffuse (already incorporating skip connections)')
print('\n💡 For U-Net segmentation, the bottleneck usually gives the best insights!')

### Which Layer Should You Use?

**General Guidelines:**

1. **Encoder Layers (enc1, enc2, enc3):**
   - Show low-level features (edges, corners, textures)
   - High spatial resolution but less semantic meaning
   - Good for understanding "what features are detected"

2. **Bottleneck (best choice!):**
   - Balance of semantic and spatial information
   - Shows "what regions matter for object recognition"
   - Usually most interpretable

3. **Decoder Layers:**
   - Already incorporating skip connections
   - Less clear attribution (multiple information sources)
   - Good for understanding skip connection contributions

**For this lab, we'll stick with the bottleneck!**

---

# PART 7: Discussion & Insights

## Key Findings from GradCAM Analysis

Based on your explorations above, let's summarize what we learned:

### Questions for Discussion:

**1. Does U-Net Focus on the Right Features?**
   - Did GradCAM show attention on object boundaries?
   - Or did it focus on background/noise?
   - Compare your observations across multiple examples

**2. Can We Trust the Model?**
   - Are predictions right for the right reasons?
   - Or could the model be using shortcuts?
   - What would make you trust (or not trust) this model in a clinical setting?

**3. How Does GradCAM Help Debug Errors?**
   - Look back at the worst prediction
   - Did GradCAM reveal why it failed?
   - How would you use this information to improve the model?

**4. Limitations of GradCAM**
   - What information does GradCAM NOT show you?
   - When might GradCAM be misleading?
   - What other explainability methods might complement GradCAM?

---

## Real-World Application

### Example: Dermatology AI

Remember the skin cancer classifier that learned rulers = malignant?

**How GradCAM Would Have Caught This:**
- GradCAM would show bright spots on rulers, not lesions
- Doctors would immediately see the model looking at wrong features
- Problem caught BEFORE clinical deployment

**This is why explainability is essential in medical AI!**

---

## Practical Takeaways

✅ **Always visualize what your model "sees"**
✅ **Don't trust metrics alone** - high accuracy ≠ correct reasoning
✅ **Use explainability to debug** - understand failures, not just count them
✅ **Communicate with stakeholders** - show doctors WHERE the model looks
✅ **Iterate based on insights** - fix data/model based on what you learn

---

# PART 8: Extensions & Next Steps

## What You Learned Today:

1. ✅ Why explainability matters in medical AI
2. ✅ How GradCAM works (gradients → weights → heatmap)
3. ✅ How to implement GradCAM for segmentation models
4. ✅ How to interpret GradCAM visualizations
5. ✅ How to use explainability to debug model errors


## Further Reading

**Papers:**
- GradCAM: Selvaraju et al., "Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization" (ICCV 2017)
- GradCAM++: Chattopadhay et al., "Grad-CAM++: Generalized Gradient-based Visual Explanations" (WACV 2018)
- Medical Imaging: Graziani et al., "Regression Concept Vectors for Bidirectional Explanations in Histopathology" (MICCAI 2018)

**Resources:**
- Distill.pub - Great visual explanations of interpretability
- Captum (PyTorch interpretability library)
- LIME and SHAP libraries for alternative explanations

---

# PART 9: Save Your Work

## Exporting GradCAM Visualizations

In [ ]:
# Save GradCAM results for your report
import os

# Create output directory
os.makedirs('gradcam_results', exist_ok=True)

# Save a few examples
n_save = 5
indices_to_save = np.random.choice(len(X_test), n_save, replace=False)

for i, idx in enumerate(indices_to_save):
    image = X_test[idx]
    mask = y_test[idx]
    pred = predictions[idx]
    
    # Generate GradCAM
    input_tensor = torch.from_numpy(image).unsqueeze(0).unsqueeze(0).to(device)
    input_tensor.requires_grad = True
    cam = gradcam.generate_cam(input_tensor)
    
    # Create visualization
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    
    axes[0].imshow(image, cmap='gray')
    axes[0].set_title('Input')
    axes[0].axis('off')
    
    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title('Ground Truth')
    axes[1].axis('off')
    
    axes[2].imshow(pred, cmap='gray', vmin=0, vmax=1)
    axes[2].set_title('Prediction')
    axes[2].axis('off')
    
    axes[3].imshow(cam, cmap='jet', vmin=0, vmax=1)
    axes[3].set_title('GradCAM')
    axes[3].axis('off')
    
    overlay = overlay_heatmap(image, cam, alpha=0.5)
    axes[4].imshow(overlay)
    axes[4].set_title('Overlay')
    axes[4].axis('off')
    
    plt.suptitle(f'GradCAM Analysis - Sample {i+1}', fontsize=14)
    plt.tight_layout()
    
    # Save
    plt.savefig(f'gradcam_results/sample_{i+1}.png', dpi=150, bbox_inches='tight')
    plt.close()

print(f'✓ Saved {n_save} GradCAM visualizations to gradcam_results/')
print('  You can use these for your report or presentation!')

---

# Summary

## What We Accomplished Today:

1. ✅ Loaded trained U-Net from last class
2. ✅ Implemented GradCAM from scratch
3. ✅ Applied GradCAM to test images
4. ✅ Analyzed good vs bad predictions
5. ✅ Explored different target layers
6. ✅ Discussed real-world implications

## Key Insight:

**Explainability is not optional in medical AI - it's essential!**

GradCAM helps us:
- Understand model decisions
- Debug errors
- Build trust with clinicians
- Meet regulatory requirements
- Improve model design